In [2]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import os

In [3]:
# Load data
benign_train = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/benign.npy')
harmful_train = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/harmful.npy')
benign_test = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/test/benign.npy')
harmful_test = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/test/harmful.npy')

# Create labels (0 for benign, 1 for harmful)
y_train = np.concatenate([
    np.ones(len(benign_train)),
    np.zeros(len(harmful_train))
])
y_test = np.concatenate([
    np.ones(len(benign_test)),
    np.zeros(len(harmful_test))
])

# Stack features
X_train = np.vstack([benign_train, harmful_train])
X_test = np.vstack([benign_test, harmful_test])
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [4]:
def classify(classifier, name):
        
    # Train on full training set
    classifier.fit(X_train, y_train)

    # Evaluate on test set (only once!)
    y_pred = classifier.predict(X_test)
    accuracy = classifier.score(X_test, y_test)
    print(f"\nTest Set Accuracy: {accuracy:.4f}")
    print("\nTest Set Results:")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Harmful']))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    joblib.dump(classifier, f'../classifiers/{name}')


In [5]:
def classify_ratio(classifier, name):
        
    # Load the saved classifier
    # classifier = joblib.load('../classifiers/model_classifier.pkl')

    # Load your data
    ratios = [x/20 for x in range(7)]
    for ratio in ratios:
            
        csv_file = f'../data/tool_dataset/beavertails/data/330k/test/sampled/ratio_{ratio}.csv'  # Replace with your CSV filename
        npy_file = f'../data/tool_dataset/beavertails/data/330k/test/sampled/ratio_{ratio}.npy'  # Replace with your NPY filename

        df = pd.read_csv(csv_file)
        representations = np.load(npy_file)

        # Verify shapes match
        assert len(df) == len(representations), f"Mismatch: CSV has {len(df)} rows, NPY has {len(representations)} rows"

        # Classify all representations
        predictions = classifier.predict(representations)
        # prediction_probs = classifier.predict_proba(representations)

        # Add predictions to dataframe
        df['predicted_label'] = predictions

        # Assuming your ground truth label column is named 'label' or 'ground_truth'
        # Update this to match your actual column name
        label_column = 'is_safe'  # Change this to your actual label column name

        if label_column in df.columns:
            # Calculate classification metrics
            print("=" * 60)
            print("CLASSIFICATION PERFORMANCE")
            print("=" * 60)
            
            accuracy = accuracy_score(df[label_column], df['predicted_label'])
            print(f"\nOverall Accuracy: {accuracy:.4f}\n")
            
            print("Classification Report:")
            print(classification_report(df[label_column], df['predicted_label']))
            
            print("\nConfusion Matrix:")
            print(confusion_matrix(df[label_column], df['predicted_label']))
            print()
        else:
            print(f"Warning: Label column '{label_column}' not found in CSV")
            print(f"Available columns: {df.columns.tolist()}")

        # Filter based on predictions
        # Example 1: Keep only rows predicted as class 1
        filtered_df_class1 = df[df['predicted_label'] == 1].copy()

        output_path = f'../data/tool_dataset/beavertails/data/330k/test/filtered/{name}/ratio_{ratio}.csv'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        # Save filtered results
        filtered_df_class1.to_csv(output_path, index=False)

        print("=" * 60)
        print("FILTERING RESULTS")
        print("=" * 60)
        print(f"Original dataset: {len(df)} rows")
        print(f"Filtered (class 1): {len(filtered_df_class1)} rows")


        # Show prediction distribution
        print("\n" + "=" * 60)
        print("PREDICTION DISTRIBUTION")
        print("=" * 60)
        print("\nPredicted class counts:")
        print(df['predicted_label'].value_counts().sort_index())

In [12]:
from sklearn.svm import SVC

svc = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('classifier', SVC(random_state=42))
])


classify(svc, 'pca_svm.pkl')


Test Set Accuracy: 0.8428

Test Set Results:
              precision    recall  f1-score   support

      Benign       0.87      0.86      0.86      1733
     Harmful       0.81      0.83      0.82      1288

    accuracy                           0.84      3021
   macro avg       0.84      0.84      0.84      3021
weighted avg       0.84      0.84      0.84      3021


Confusion Matrix:
[[1483  250]
 [ 225 1063]]


In [6]:
svc = joblib.load("../classifiers/pca_svm.pkl")
classify_ratio(svc, 'pca_svm')

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7977

Classification Report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         0
         1.0       1.00      0.80      0.89      3000

    accuracy                           0.80      3000
   macro avg       0.50      0.40      0.44      3000
weighted avg       1.00      0.80      0.89      3000


Confusion Matrix:
[[   0    0]
 [ 607 2393]]



/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2393 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     607
1.0    2393
Name: count, dtype: int64
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.8057

Classification Report:
              precision    recall  f1-score   support

       False       0.19      0.89      0.31       150
        True       0.99      0.80      0.89      2850

    accuracy                           0.81      3000
   macro avg       0.59      0.85      0.60      3000
weighted avg       0.95      0.81      0.86      3000


Confusion Matrix:
[[ 134   16]
 [ 567 2283]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2299 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     701
1.0    2299
Name: count, dtype: int64
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.8103

Classification Report:
              precision    recall  f1-score   support

       False       0.33      0.88

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os
import pandas as pd


class PyTorchClassifierWrapper:
    """Wrapper to make PyTorch models compatible with scikit-learn interface"""
    
    def __init__(self, model, epochs=100, batch_size=32, lr=0.001, device=None):
        self.model = model
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = None
        self.classes_ = None
        
    def fit(self, X, y):
        """Train the model"""
        # Convert to tensors - force float32
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        X_tensor = torch.from_numpy(X).to(self.device)
        y_tensor = torch.from_numpy(y).to(self.device)
        
        # Store classes
        self.classes_ = np.unique(y)
        
        # Create dataset and dataloader
        dataset = TensorDataset(X_tensor, y_tensor)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        # Move model to device
        self.model = self.model.to(self.device)
        
        # Initialize optimizer
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        
        # Training loop
        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            correct = 0
            total = 0
            
            for batch_X, batch_y in dataloader:
                # Forward pass
                self.optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = self.criterion(outputs, batch_y)
                
                # Backward pass
                loss.backward()
                self.optimizer.step()
                
                # Track metrics
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
            
            # Print progress every 10 epochs
            if (epoch + 1) % 10 == 0:
                avg_loss = total_loss / len(dataloader)
                accuracy = 100 * correct / total
                print(f'Epoch [{epoch+1}/{self.epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')
        
        return self
    
    def predict(self, X):
        """Predict classes"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor)
            _, predicted = torch.max(outputs.data, 1)
            return predicted.cpu().numpy()
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor)
            probs = torch.softmax(outputs, dim=1)
            return probs.cpu().numpy()
    
    def score(self, X, y):
        """Calculate accuracy"""
        predictions = self.predict(X)
        return accuracy_score(y, predictions)


# Example Neural Network Architecture
class SimpleNN(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super(SimpleNN, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.7),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            
            nn.Linear(32, 2)  # 2 classes for binary classification
        )
    
    def forward(self, x):
        return self.network(x)


def classify(classifier, name):
    """Train and evaluate classifier"""
    
    # Train on full training set
    classifier.fit(X_train, y_train)
    
    # Evaluate on test set (only once!)
    y_pred = classifier.predict(X_test)
    accuracy = classifier.score(X_test, y_test)
    print(f"\nTest Set Accuracy: {accuracy:.4f}")
    print("\nTest Set Results:")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Harmful']))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    joblib.dump(classifier, f'../classifiers/{name}')
    return classifier


def classify_ratio(classifier, name):
    """Classify and filter data by ratio"""
    
    ratios = [x/20 for x in range(7)]
    for ratio in ratios:
        csv_file = f'../data/tool_dataset/beavertails/data/330k/test/sampled/ratio_{ratio}.csv'
        npy_file = f'../data/tool_dataset/beavertails/data/330k/test/sampled/ratio_{ratio}.npy'
        
        df = pd.read_csv(csv_file)
        representations = np.load(npy_file)
        
        # Verify shapes match
        assert len(df) == len(representations), f"Mismatch: CSV has {len(df)} rows, NPY has {len(representations)} rows"
        
        # Classify all representations
        predictions = classifier.predict(representations)
        prediction_probs = classifier.predict_proba(representations)
        
        # Add predictions to dataframe
        df['predicted_label'] = predictions
        
        label_column = 'is_safe'
        
        if label_column in df.columns:
            # Calculate classification metrics
            print("=" * 60)
            print("CLASSIFICATION PERFORMANCE")
            print("=" * 60)
            
            accuracy = accuracy_score(df[label_column], df['predicted_label'])
            print(f"\nOverall Accuracy: {accuracy:.4f}\n")
            
            print("Classification Report:")
            print(classification_report(df[label_column], df['predicted_label']))
            
            print("\nConfusion Matrix:")
            print(confusion_matrix(df[label_column], df['predicted_label']))
            print()
        else:
            print(f"Warning: Label column '{label_column}' not found in CSV")
            print(f"Available columns: {df.columns.tolist()}")
        
        # Filter based on predictions
        filtered_df_class1 = df[df['predicted_label'] == 1].copy()
        output_path = f'../data/tool_dataset/beavertails/data/330k/test/filtered/{name}/ratio_{ratio}.csv'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Save filtered results
        filtered_df_class1.to_csv(output_path, index=False)
        
        print("=" * 60)
        print("FILTERING RESULTS")
        print("=" * 60)
        print(f"Original dataset: {len(df)} rows")
        print(f"Filtered (class 1): {len(filtered_df_class1)} rows")
        
        # Show prediction distribution
        print("\n" + "=" * 60)
        print("PREDICTION DISTRIBUTION")
        print("=" * 60)
        print("\nPredicted class counts:")
        print(df['predicted_label'].value_counts().sort_index())



# 1. Create your PyTorch model
input_dim = X_train.shape[1]  # Number of features
model = SimpleNN(input_dim=input_dim, dropout=0.3)

# 2. Wrap it
classifier = PyTorchClassifierWrapper(
    model=model,
    epochs=100,
    batch_size=32,
    lr=0.001
)

# 3. Use with your existing functions!
classify(classifier, 'pytorch_model')
classify_ratio(classifier, 'pytorch_model')

# ========================================================================
# ALTERNATIVE: More complex architecture
# ========================================================================

# class DeepNN(nn.Module):
#     def __init__(self, input_dim):
#         super(DeepNN, self).__init__()
        
#         self.fc1 = nn.Linear(input_dim, 256)
#         self.bn1 = nn.BatchNorm1d(256)
#         self.dropout1 = nn.Dropout(0.4)
        
#         self.fc2 = nn.Linear(256, 128)
#         self.bn2 = nn.BatchNorm1d(128)
#         self.dropout2 = nn.Dropout(0.3)
        
#         self.fc3 = nn.Linear(128, 64)
#         self.bn3 = nn.BatchNorm1d(64)
#         self.dropout3 = nn.Dropout(0.2)
        
#         self.fc4 = nn.Linear(64, 32)
#         self.fc5 = nn.Linear(32, 2)
    
#     def forward(self, x):
#         x = torch.relu(self.bn1(self.fc1(x)))
#         x = self.dropout1(x)
        
#         x = torch.relu(self.bn2(self.fc2(x)))
#         x = self.dropout2(x)
        
#         x = torch.relu(self.bn3(self.fc3(x)))
#         x = self.dropout3(x)
        
#         x = torch.relu(self.fc4(x))
#         x = self.fc5(x)
#         return x

# # Use the deeper model
# deep_model = DeepNN(input_dim=input_dim)
# deep_classifier = PyTorchClassifierWrapper(
#     model=deep_model,
#     epochs=150,
#     batch_size=64,
#     lr=0.0005
# )

# classify(deep_classifier, 'deep_pytorch_model')

Epoch [10/100], Loss: 0.3545, Accuracy: 84.33%
Epoch [20/100], Loss: 0.3319, Accuracy: 85.40%
Epoch [30/100], Loss: 0.3168, Accuracy: 85.96%
Epoch [40/100], Loss: 0.3071, Accuracy: 86.40%
Epoch [50/100], Loss: 0.2992, Accuracy: 86.59%
Epoch [60/100], Loss: 0.2917, Accuracy: 86.85%
Epoch [70/100], Loss: 0.2865, Accuracy: 86.93%
Epoch [80/100], Loss: 0.2818, Accuracy: 87.12%
Epoch [90/100], Loss: 0.2766, Accuracy: 87.18%
Epoch [100/100], Loss: 0.2748, Accuracy: 87.25%

Test Set Accuracy: 0.8888

Test Set Results:
              precision    recall  f1-score   support

      Benign       0.93      0.87      0.90      1733
     Harmful       0.84      0.91      0.87      1288

    accuracy                           0.89      3021
   macro avg       0.89      0.89      0.89      3021
weighted avg       0.89      0.89      0.89      3021


Confusion Matrix:
[[1516  217]
 [ 119 1169]]
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.8237

Classification Report:
              precision    recall

/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.8263

Classification Report:
              precision    recall  f1-score   support

       False       0.20      0.81      0.32       150
        True       0.99      0.83      0.90      2850

    accuracy                           0.83      3000
   macro avg       0.59      0.82      0.61      3000
weighted avg       0.95      0.83      0.87      3000


Confusion Matrix:
[[ 121   29]
 [ 492 2358]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2387 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0     613
1    2387
Name: count, dtype: int64
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.8277

Classification Report:
              precision    recall  f1-score   support

       False       0.35      0.82      0.49       300
        True       0.98      0.83      0.90      2700

    accuracy                           0.83      3000
   macro avg       0.66      0.82      0.69      3000
weighted avg

In [5]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Complete pipeline with Random Forest
# random_forest = Pipeline([
#     ('scaler', StandardScaler()),
#     ('pca', PCA(n_components=0.95)),
#     ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
# ])

# # Or with Gradient Boosting
# gradient_boosting = Pipeline([
#     ('scaler', StandardScaler()),
#     ('pca', PCA(n_components=0.95)),
#     ('classifier', GradientBoostingClassifier(random_state=42))
# ])


# classify(random_forest, 'pca_randomforest.pkl')

In [7]:
gb = joblib.load("../classifiers/pca_GB.pkl")
classify_ratio(gb, "pca_GradientBoosting")

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7690

Classification Report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         0
         1.0       1.00      0.77      0.87      3000

    accuracy                           0.77      3000
   macro avg       0.50      0.38      0.43      3000
weighted avg       1.00      0.77      0.87      3000


Confusion Matrix:
[[   0    0]
 [ 693 2307]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2307 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     693
1.0    2307
Name: count, dtype: int64


/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7770

Classification Report:
              precision    recall  f1-score   support

       False       0.17      0.87      0.28       150
        True       0.99      0.77      0.87      2850

    accuracy                           0.78      3000
   macro avg       0.58      0.82      0.57      3000
weighted avg       0.95      0.78      0.84      3000


Confusion Matrix:
[[ 130   20]
 [ 649 2201]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2221 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     779
1.0    2221
Name: count, dtype: int64
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7830

Classification Report:
              precision    recall  f1-score   support

       False       0.30      0.86      0.44       300
        True       0.98      0.77      0.87      2700

    accuracy                           0.78      3000
   macro avg       0.64      0.82      0.65      3000
weighted

In [11]:
classify_ratio(random_forest, 'pca_randomforest')

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7187

Classification Report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         0
         1.0       1.00      0.72      0.84      3000

    accuracy                           0.72      3000
   macro avg       0.50      0.36      0.42      3000
weighted avg       1.00      0.72      0.84      3000


Confusion Matrix:
[[   0    0]
 [ 844 2156]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2156 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     844
1.0    2156
Name: count, dtype: int64


/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7307

Classification Report:
              precision    recall  f1-score   support

       False       0.14      0.89      0.25       150
        True       0.99      0.72      0.84      2850

    accuracy                           0.73      3000
   macro avg       0.57      0.80      0.54      3000
weighted avg       0.95      0.73      0.81      3000


Confusion Matrix:
[[ 133   17]
 [ 791 2059]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2076 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     924
1.0    2076
Name: count, dtype: int64
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7383

Classification Report:
              precision    recall  f1-score   support

       False       0.26      0.89      0.40       300
        True       0.98      0.72      0.83      2700

    accuracy                           0.74      3000
   macro avg       0.62      0.80      0.62      3000
weighted

In [13]:
lr = joblib.load('../classifiers/pca_LR.pkl')
classify_ratio(lr, 'pca_lr')

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7750

Classification Report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         0
         1.0       1.00      0.78      0.87      3000

    accuracy                           0.78      3000
   macro avg       0.50      0.39      0.44      3000
weighted avg       1.00      0.78      0.87      3000


Confusion Matrix:
[[   0    0]
 [ 675 2325]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2325 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     675
1.0    2325
Name: count, dtype: int64


/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/michael920403/repos/fine-tuning-attack/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7847

Classification Report:
              precision    recall  f1-score   support

       False       0.17      0.89      0.29       150
        True       0.99      0.78      0.87      2850

    accuracy                           0.78      3000
   macro avg       0.58      0.83      0.58      3000
weighted avg       0.95      0.78      0.84      3000


Confusion Matrix:
[[ 133   17]
 [ 629 2221]]

FILTERING RESULTS
Original dataset: 3000 rows
Filtered (class 1): 2238 rows

PREDICTION DISTRIBUTION

Predicted class counts:
predicted_label
0.0     762
1.0    2238
Name: count, dtype: int64
CLASSIFICATION PERFORMANCE

Overall Accuracy: 0.7890

Classification Report:
              precision    recall  f1-score   support

       False       0.30      0.86      0.45       300
        True       0.98      0.78      0.87      2700

    accuracy                           0.79      3000
   macro avg       0.64      0.82      0.66      3000
weighted

In [14]:
classify(gradient_boosting, 'pca_GB.pkl')


Test Set Accuracy: 0.8044

Test Set Results:
              precision    recall  f1-score   support

      Benign       0.83      0.82      0.83      1733
     Harmful       0.77      0.78      0.77      1288

    accuracy                           0.80      3021
   macro avg       0.80      0.80      0.80      3021
weighted avg       0.80      0.80      0.80      3021


Confusion Matrix:
[[1429  304]
 [ 287 1001]]
